In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

In [2]:
df = pd.read_csv('Duplicate Question.csv')
print(df.shape)
df.sample(5)

(404290, 6)


,id,qid1,qid2,question1,question2,is_duplicate
389481,389481,121207,478069,How do I manually upgrade my Android 4.4.2 to ...,How do I upgrade my andriod 4.2.2 to 5.1?,0
140845,140845,223748,223749,Where do I get songs like Moon River?,"What's the meaning of the lyrics in the song ""...",0
361259,361259,284425,491078,What kind of employment can an H4 visa holder ...,Can H4 visa holders take any unpaid internships?,0
261975,261975,378219,378220,What are some good Salads recipes?,How can we mitigate the pilot pollution from o...,0
66688,66688,49437,1063,Is World War 3 coming?,Are we heading toward World War 3?,1


In [3]:
df = df.sample(30000)
df.shape

(30000, 6)

In [4]:
df.isnull().sum()

id              0
qid1            0
qid2            0
question1       0
question2       0
is_duplicate    0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.sample()

,id,qid1,qid2,question1,question2,is_duplicate
122883,122883,198797,198798,How capable is the M60 MBT in modern combat?,Just how fast would an M4 Sherman get wiped ou...,0


# Extracting the required columns to for BoW

In [7]:
ques_df =  df.iloc[:,3:5]
ques_df.head()

,question1,question2
128471,Why did Ryan Lochte lie about getting robbed a...,"In the 5th book, why does Harry say that he co..."
64342,Why do parents want their kids to become a doc...,Why do most Asian parents want their children ...
247041,How do you get rid of a swarm of bugs in your ...,I am first year student at bcom hons in delhi ...
193729,What are the best songs to put on a running/wo...,What are the some of the best songs for the gy...
366149,What are the best and quickest ways to reduce ...,What is the effective ways of reducing stress?


# Applying Bag of Words

In [8]:
from sklearn.feature_extraction.text import CountVectorizer 
cv = CountVectorizer(max_features=3000) 


# Merging the q1 and q2 together so that we can apply the BoW together on both questions

In [9]:
# Merging the questions in single list 
ques = list(ques_df['question1'].to_list() + ques_df['question2'].to_list()) 
ques

['Why did Ryan Lochte lie about getting robbed at Rio 2016 when he could have just kept quiet?',
 'Why do parents want their kids to become a doctor?',
 'How do you get rid of a swarm of bugs in your basement?',
 'What are the best songs to put on a running/workout playlist?',
 'What are the best and quickest ways to reduce stress?',
 'How can I delete my blocked Facebook account by gmail account?',
 'Is it safe to buy an unlocked phone from eBay India? What if I receive a defective product as the seller has specified no refund? Should I buy it an HTC One m8 Verizon unlocked version?',
 'Where is the safest place to shoot somebody so he/she does not die immediately?',
 'How will Trump or Clinton presidency affect foreign students?',
 'How good is to work at Google?',
 'Does Arrakis work well with EC2?',
 'What is the difference between the ThinkPad and IdeaPad?',
 'What are the health benefits of eating oxtails?',
 "What are the best new products in technology that people don't know ab

In [10]:
# Creating the Arrays of  questions q1 and q2 
q1_arr, q2_arr = np.vsplit(cv.fit_transform(ques).toarray(),2)
q1_arr.shape, q2_arr.shape

((30000, 3000), (30000, 3000))

In [11]:
# Merging the q1_arr and q2_arr 
temp_d1 = pd.DataFrame(q1_arr, index=ques_df.index)
temp_d2 = pd.DataFrame(q2_arr, index = ques_df.index)
temp_df = pd.concat([temp_d1,temp_d2],axis=1)
print(temp_df.shape)

(30000, 6000)


In [12]:
temp_df.head()

,0,1,2,3,4,5,6,7,8,9,...,2990,2991,2992,2993,2994,2995,2996,2997,2998,2999
128471,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
64342,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
247041,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
193729,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
366149,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
temp_df['is_duplicate'] = df['is_duplicate']
temp_df.sample(5)

,0,1,2,3,4,5,6,7,8,9,...,2991,2992,2993,2994,2995,2996,2997,2998,2999,is_duplicate
73244,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
144920,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
353821,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
214126,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
212610,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
from sklearn.model_selection import train_test_split 
x_train,x_test,y_train,y_test = train_test_split(temp_df.drop(columns='is_duplicate').values, 
                                                 temp_df.iloc[:,-1].values,test_size=0.2,random_state=1)

# Applying Algorithms to find out the Accuracy 

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score 
from sklearn.metrics import ConfusionMatrixDisplay 

rf = RandomForestClassifier() 
rf.fit(x_train,y_train) 
y_pred_rf = rf.predict(x_test) 
print('Accuracy score for Random Forest -->', accuracy_score(y_test,y_pred_rf))

Accuracy score for Random Forest --> 0.7488333333333334


In [17]:
from xgboost import XGBClassifier 

xg = XGBClassifier() 
xg.fit(x_train,y_train)
y_pred_xg = xg.predict(x_test) 
print('Accuracy score for XG Boost -->',accuracy_score(y_test,y_pred_xg))

Accuracy score for XG Boost --> 0.7298333333333333
